# Import Libraries

In [8]:
import pandas as pd
import requests
from PIL import Image
from io import BytesIO
import os
from tqdm import tqdm

# Load Cleaned Dataset

In [9]:
df = pd.read_csv("../data/processed/cleaned_electronics_products.csv")

# Verify Dataset

In [10]:
df.head()

,Unnamed: 0,name,main_category,sub_category,image,link,ratings,no_of_ratings,discount_price,actual_price
0,0,"Redmi 10 Power (Power Black, 8GB RAM, 128GB St...","tv, audio & cameras",All Electronics,https://m.media-amazon.com/images/I/81eM15lVcJ...,https://www.amazon.in/Redmi-Power-Black-128GB-...,4.0,965,"₹10,999","₹18,999"
1,1,"OnePlus Nord CE 2 Lite 5G (Blue Tide, 6GB RAM,...","tv, audio & cameras",All Electronics,https://m.media-amazon.com/images/I/71AvQd3Vzq...,https://www.amazon.in/OnePlus-Nord-Lite-128GB-...,4.3,"113,956","₹18,999","₹19,999"
2,2,OnePlus Bullets Z2 Bluetooth Wireless in Ear E...,"tv, audio & cameras",All Electronics,https://m.media-amazon.com/images/I/51UhwaQXCp...,https://www.amazon.in/Oneplus-Bluetooth-Wirele...,4.2,"90,304","₹1,999","₹2,299"
3,3,"Samsung Galaxy M33 5G (Mystique Green, 6GB, 12...","tv, audio & cameras",All Electronics,https://m.media-amazon.com/images/I/81I3w4J6yj...,https://www.amazon.in/Samsung-Mystique-Storage...,4.1,"24,863","₹15,999","₹24,999"
4,4,"OnePlus Nord CE 2 Lite 5G (Black Dusk, 6GB RAM...","tv, audio & cameras",All Electronics,https://m.media-amazon.com/images/I/71V--WZVUI...,https://www.amazon.in/OnePlus-Nord-Black-128GB...,4.3,"113,956","₹18,999","₹19,999"


In [11]:
df.shape

(9600, 10)

# Create Images Folder

In [12]:
image_folder = "../data/images"

os.makedirs(image_folder, exist_ok=True)

print("Images folder is ready!")

Images folder is ready!


# Initialize Download Counters

In [13]:
successful_downloads = 0
failed_downloads = 0
failed_images = []

# Test Download (First 5 Images)

In [14]:
for index, row in df.head(5).iterrows():
    try:
        image_url = row["image"]

        response = requests.get(image_url, timeout=10)
        response.raise_for_status()

        image = Image.open(BytesIO(response.content))

        image_path = os.path.join(image_folder, f"{index}.jpg")

        image.save(image_path)

        print(f"✅ Downloaded Image {index}")

    except Exception as e:
        print(f"❌ Failed Image {index}: {e}")

✅ Downloaded Image 0
✅ Downloaded Image 1
✅ Downloaded Image 2
✅ Downloaded Image 3
✅ Downloaded Image 4


# Download All Product Images

In [ ]:
successful_downloads = 0
failed_downloads = 0
failed_images = []

for index, row in tqdm(df.iterrows(), total=len(df), desc="Downloading Images"):

    image_path = os.path.join(image_folder, f"{index}.jpg")

    # Skip if already downloaded
    if os.path.exists(image_path):
        successful_downloads += 1
        continue

    try:
        image_url = row["image"]

        response = requests.get(image_url, timeout=10)
        response.raise_for_status()

        image = Image.open(BytesIO(response.content))

        image.save(image_path)

        successful_downloads += 1

    except Exception as e:

        failed_downloads += 1

        failed_images.append({
            "index": index,
            "product_name": row["name"],
            "image_url": image_url,
            "error": str(e)
        })

KeyboardInterrupt: 

In [ ]:
import os
import pandas as pd
import requests
from PIL import Image
from io import BytesIO
from tqdm import tqdm

# Paths relative to the notebook file
image_folder = "../data/images"
dataset_path = "../data/processed/cleaned_electronics_products.csv"

os.makedirs(image_folder, exist_ok=True)

df = pd.read_csv(dataset_path)

remaining = []
for index, row in df.iterrows():
    image_path = os.path.join(image_folder, f"{index}.jpg")
    if not os.path.exists(image_path):
        remaining.append((index, row))

print(f"Existing images: {len(df) - len(remaining)}, remaining: {len(remaining)}")

successful_downloads = 0
failed_downloads = 0
failed_images = []

with requests.Session() as session:
    for index, row in tqdm(remaining, desc="Resuming download", total=len(remaining)):
        image_url = row["image"]
        image_path = os.path.join(image_folder, f"{index}.jpg")

        try:
            response = session.get(image_url, timeout=10)
            response.raise_for_status()

            image = Image.open(BytesIO(response.content))
            image.save(image_path)

            successful_downloads += 1
        except Exception as e:
            failed_downloads += 1
            failed_images.append({
                "index": index,
                "product_name": row.get("name", ""),
                "image_url": image_url,
                "error": str(e)
            })

print("\nResume complete")
print("New successful downloads:", successful_downloads)
print("Failed downloads:", failed_downloads)
print("Total images present:", len(os.listdir(image_folder)))

if failed_images:
    failed_df = pd.DataFrame(failed_images)
    display(failed_df.head(20))
